In [1]:
import pandas as pd
from ipynb.fs.defs.functions import clean_and_sort, validate_df

In [2]:
# CSV-Datei einlesen
df = pd.read_csv("../data/raw/2024/22517-02i.csv", sep=";", encoding="latin1", skiprows=6)
# Quelle: https://www.landesdatenbank.nrw.de/


In [3]:
# Überblick:
print(df.head(10))
print(df.info())

  Unnamed: 0 Unnamed: 1                            Unnamed: 2  \
0        NaN        NaN                                   NaN   
1        NaN        NaN                                   NaN   
2       2024         05                   Nordrhein-Westfalen   
3       2024        051          Düsseldorf, Regierungsbezirk   
4       2024      05111               Düsseldorf, krfr. Stadt   
5       2024      05112                 Duisburg, krfr. Stadt   
6       2024      05113                    Essen, krfr. Stadt   
7       2024      05114                  Krefeld, krfr. Stadt   
8       2024      05116          Mönchengladbach, krfr. Stadt   
9       2024      05117      Mülheim an der Ruhr, krfr. Stadt   

                              Insgesamt  \
0  In Anspruch genommene erzieh. Hilfen   
1                                Anzahl   
2                                311197   
3                                 89998   
4                                 11264   
5                         

In [4]:
# nach Kreisen und kreisfreien Städten und in Anspruch genommenen (35a) Hilfen filtern

df = df[['Unnamed: 2', 'Insgesamt','Eingliederungshilfe für seelisch behinderte junge Menschen § 35a SGB VIII']]
df = df.dropna(axis=1, how="all")
df = df.dropna(axis=0, how="all")
df = df.rename(columns={"Unnamed: 2": "Name", "Eingliederungshilfe für seelisch behinderte junge Menschen § 35a SGB VIII": "Anzahl 35a Hilfen"})
df = df[df["Name"].str.contains("Kreis|krfr. Stadt|Städteregion", case=False,na=False)]
df = df[~df["Name"].str.contains("Kreisjugendamt", case=False, na=False)]

# Object-Types parsen
df['Anzahl 35a Hilfen'] = pd.to_numeric(df['Anzahl 35a Hilfen'], errors='coerce').astype("Int64")
df['Insgesamt'] = pd.to_numeric(df['Insgesamt'], errors='coerce').astype("Int64")

In [5]:
# fehlende Werte finden
print(df[df["Anzahl 35a Hilfen"].isna()])
print(df[df["Insgesamt"].isna()])

                       Name  Insgesamt  Anzahl 35a Hilfen
6        Essen, krfr. Stadt       3894               <NA>
54      Aachen, krfr. Stadt       <NA>               <NA>
66            Aachen, Kreis       <NA>               <NA>
                       Name  Insgesamt  Anzahl 35a Hilfen
54      Aachen, krfr. Stadt       <NA>               <NA>
66            Aachen, Kreis       <NA>               <NA>


In [6]:
# Aachen ist mittlerweile Städteregion, daher wird alles außer Städteregion gelöscht
# für Essen wurden keine validen Daten geliefert

df["Name"] = df["Name"].str.strip()
df = df[df["Name"] != "Aachen, krfr. Stadt"]
df = df[df["Name"] != "Aachen, Kreis"]
df = df[df["Name"] != "Essen, krfr. Stadt"]

In [7]:
df.head()

,Name,Insgesamt,Anzahl 35a Hilfen
4,"Düsseldorf, krfr. Stadt",11264,523
5,"Duisburg, krfr. Stadt",10763,1327
7,"Krefeld, krfr. Stadt",5291,495
8,"Mönchengladbach, krfr. Stadt",4860,375
9,"Mülheim an der Ruhr, krfr. Stadt",2711,659


In [8]:
df.tail()

,Name,Insgesamt,Anzahl 35a Hilfen
206,Märkischer Kreis,7074,627
215,"Olpe, Kreis",2450,388
217,"Siegen-Wittgenstein, Kreis",4324,826
220,"Soest, Kreis",5429,843
225,"Unna, Kreis",7903,1128


In [ ]:
# Typ (Kreis/kreisfreie Stadt) extrahieren
kreisfreie_keywords = ["krfr. Stadt", "kreisfreie Stadt", "Stadt"]
kreis_keywords = ["Kreis", "kreis"]

def extract(raw: str):
    raw = str(raw).strip()

    if any(k in raw for k in kreisfreie_keywords):
        return raw, "Kreisfreie Stadt"

    if any(k in raw for k in kreis_keywords):     
        return raw, "Kreis"

    return raw, "Unbekannt"

# Name/Typ erzeugen
df[["Name", "Typ"]] = df["Name"].apply(lambda x: pd.Series(extract(x)))
print((df["Typ"] == "Unbekannt").sum())

In [ ]:
# Strings parsen und anpassen
df = clean_and_sort(df, "Typ", "Insgesamt", "Anzahl 35a Hilfen")
df.loc[df["Name"] == "Aachen", "Typ"] = "Städteregion"

In [ ]:
validate_df(
    df,
    not_null=["Insgesamt", "Anzahl 35a Hilfen"],
    non_negative=["Insgesamt", "Anzahl 35a Hilfen"],
    numeric=["Insgesamt", "Anzahl 35a Hilfen"],
    key_cols=["Name"],
)

In [ ]:
# saubere Tabelle abspeichern
df.to_csv("../data/processed/hilfen_2024.csv", index=False)